In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
!pip install pycocotools tqdm requests


In [ ]:
import os

BASE = "/content/drive/MyDrive/COCO_FULL_ZIPS"
os.makedirs(BASE, exist_ok=True)

print("Saving files to:", BASE)


Saving files to: /content/drive/MyDrive/COCO_FULL_ZIPS


In [ ]:
import requests

def download_zip(url, save_path):
    if os.path.exists(save_path):
        print(f"{os.path.basename(save_path)} already exists, skipping.")
        return

    print(f"Downloading {os.path.basename(save_path)}...")
    r = requests.get(url, stream=True)
    r.raise_for_status()

    with open(save_path, "wb") as f:
        for chunk in r.iter_content(chunk_size=1024 * 1024):  # 1MB chunks
            if chunk:
                f.write(chunk)

    print("Done.")

urls = {
    "train2014.zip": "http://images.cocodataset.org/zips/train2014.zip",
    "val2014.zip": "http://images.cocodataset.org/zips/val2014.zip",
    "annotations_trainval2014.zip": "http://images.cocodataset.org/annotations/annotations_trainval2014.zip",
}

for name, url in urls.items():
    download_zip(url, f"{BASE}/{name}")


train2014.zip already exists, skipping.
val2014.zip already exists, skipping.
annotations_trainval2014.zip already exists, skipping.


In [ ]:
import os
import zipfile

WORK_DIR = "/content/coco"
os.makedirs(WORK_DIR, exist_ok=True)

def unzip_file(src, dst):
    if not os.path.exists(dst):
        os.makedirs(dst)
    with zipfile.ZipFile(src, 'r') as zip_ref:
        zip_ref.extractall(dst)
    print(f"Extracted {os.path.basename(src)} to {dst}")

# Paths to your downloaded zips
TRAIN_ZIP = "/content/drive/MyDrive/COCO_FULL_ZIPS/train2014.zip"
VAL_ZIP   = "/content/drive/MyDrive/COCO_FULL_ZIPS/val2014.zip"
ANN_ZIP   = "/content/drive/MyDrive/COCO_FULL_ZIPS/annotations_trainval2014.zip"

unzip_file(TRAIN_ZIP, WORK_DIR)
unzip_file(VAL_ZIP, WORK_DIR)
unzip_file(ANN_ZIP, WORK_DIR)

# Verify folder structure
!ls /content/coco

Extracted train2014.zip to /content/coco
Extracted val2014.zip to /content/coco
Extracted annotations_trainval2014.zip to /content/coco
annotations  train2014	val2014


In [ ]:
!pip install tensorflow==2.19.0 nltk pycocotools tqdm

import tensorflow as tf
import numpy as np
import os
import json
import random
from tqdm import tqdm
import nltk
from PIL import Image

nltk.download('punkt')
print("TensorFlow version:", tf.__version__)

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


TensorFlow version: 2.19.0


In [ ]:
!pip install tensorflow pycocotools tqdm requests

import os
import shutil
import zipfile
import json
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

In [ ]:
import zipfile
import os

# Paths
DRIVE_ZIP = "/content/drive/MyDrive/COCO_FULL_ZIPS"
VM_COCO = "/content/coco_dataset"
os.makedirs(VM_COCO, exist_ok=True)

# Function to unzip
def unzip_file(zip_path, extract_to):
    print(f"Unzipping {os.path.basename(zip_path)} ...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_to)
    print("Done.")

# Unzip train2014
unzip_file(f"{DRIVE_ZIP}/train2014.zip", VM_COCO)

# Unzip val2014
unzip_file(f"{DRIVE_ZIP}/val2014.zip", VM_COCO)

# Unzip annotations
unzip_file(f"{DRIVE_ZIP}/annotations_trainval2014.zip", VM_COCO)


Unzipping train2014.zip ...
Done.
Unzipping val2014.zip ...
Done.
Unzipping annotations_trainval2014.zip ...
Done.


In [ ]:
import os
print(os.listdir(VM_COCO))


In [ ]:
import json
import tensorflow as tf
import os
import numpy as np

In [ ]:
COCO_ROOT = "/content/coco_dataset"
IMG_DIR = f"{COCO_ROOT}/train2014"
VAL_IMG_DIR = f"{COCO_ROOT}/val2014"
ANN_DIR = f"{COCO_ROOT}/annotations"


In [ ]:
import os
import json

# ----------------------------
# CORRECT COCO ROOT
# ----------------------------
COCO_ROOT = "/content/drive/MyDrive/coco_dataset"
ANN_DIR = f"{COCO_ROOT}/annotations"
IMG_DIR = f"{COCO_ROOT}/train2014"

# ----------------------------
# LOAD ANNOTATIONS
# ----------------------------
with open(f"{ANN_DIR}/captions_train2014.json", "r") as f:
    coco = json.load(f)

print("Annotations loaded successfully")

# ----------------------------
# IMAGE ID → FILE NAME
# ----------------------------
id2file = {img["id"]: img["file_name"] for img in coco["images"]}

# ----------------------------
# IMAGE ID → CAPTIONS
# ----------------------------
captions = {}
for ann in coco["annotations"]:
    img_id = ann["image_id"]
    cap = f"<start> {ann['caption']} <end>"
    captions.setdefault(img_id, []).append(cap)

# ----------------------------
# BUILD TRAINING LISTS
# ----------------------------
img_paths = []
cap_list = []

for img_id in captions.keys():
    img_paths.append(id2file[img_id])   # file name only
    cap_list.append(captions[img_id])   # list of captions

# ----------------------------
# USE ONLY 20% OF DATA
# ----------------------------
n = len(img_paths) // 5
img_paths = img_paths[:n]
cap_list = cap_list[:n]

print("Images used for training:", len(img_paths))


Annotations loaded successfully
Images used for training: 16556


In [ ]:
import os


print("MyDrive contents:")
print(os.listdir("/content/drive/MyDrive"))


MyDrive contents:
['Colab Notebooks', 'coco_dataset', 'COCO_FULL_ZIPS', 'tokenizer.pkl', 'ckpt-88.data-00000-of-00001', 'ckpt-88.index', 'ckpt-89.data-00000-of-00001', 'ckpt-89.index', 'ckpt-95.data-00000-of-00001', 'ckpt-95.index', 'ckpt-96.data-00000-of-00001', 'ckpt-96.index', 'coco_checkpoints']


In [ ]:
import os
import json

# ----------------------------
# CORRECT COCO ROOT
# ----------------------------
COCO_ROOT = "/content/drive/MyDrive/coco_dataset"
ANN_DIR = f"{COCO_ROOT}/annotations"
IMG_DIR = f"{COCO_ROOT}/train2014"

# ----------------------------
# LOAD ANNOTATIONS
# ----------------------------
with open(f"{ANN_DIR}/captions_train2014.json", "r") as f:
    coco = json.load(f)

print("Annotations loaded successfully")

# ----------------------------
# IMAGE ID → FILE NAME
# ----------------------------
id2file = {img["id"]: img["file_name"] for img in coco["images"]}

# ----------------------------
# IMAGE ID → CAPTIONS
# ----------------------------
captions = {}
for ann in coco["annotations"]:
    img_id = ann["image_id"]
    cap = f"<start> {ann['caption']} <end>"
    captions.setdefault(img_id, []).append(cap)

# ----------------------------
# BUILD TRAINING LISTS
# ----------------------------
img_paths = []
cap_list = []

for img_id in captions.keys():
    img_paths.append(id2file[img_id])   # file name only
    cap_list.append(captions[img_id])   # list of captions

# ----------------------------
# USE ONLY 20% OF DATA
# ----------------------------
n = len(img_paths) // 5
img_paths = img_paths[:n]
cap_list = cap_list[:n]

print("Images used for training:", len(img_paths))

Annotations loaded successfully
Images used for training: 16556


In [ ]:
import os

print(os.listdir("/content/drive/MyDrive/coco_dataset"))

['val2014', 'annotations', 'train2014']


In [ ]:
import pickle

TOKENIZER_PATH = "/content/drive/MyDrive/tokenizer.pkl"

with open(TOKENIZER_PATH, "rb") as f:
    tokenizer = pickle.load(f)

VOCAB_SIZE = len(tokenizer.word_index) + 1
MAX_LEN = 20

print("Loaded tokenizer. VOCAB_SIZE:", VOCAB_SIZE)


Loaded tokenizer. VOCAB_SIZE: 37739


In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.sequence import pad_sequences

BATCH_SIZE = 16

def flatten_captions(nested_captions):
    flat_list = []
    for sublist in nested_captions:
        for item in sublist:
            flat_list.append(item)
    return flat_list

# Convert captions to sequences
seqs = tokenizer.texts_to_sequences(flatten_captions(cap_list))
seqs = pad_sequences(seqs, maxlen=MAX_LEN, padding='post')

# Split into input & target
input_seqs = seqs[:, :-1]
target_seqs = seqs[:, 1:]

# Repeat image paths for each caption
img_paths_flat = []
for i, caps in enumerate(cap_list):
    img_paths_flat.extend([img_paths[i]] * len(caps))

# FIX: Redefine IMG_DIR locally to point to the correct location on the VM
# where images were unzipped. The global IMG_DIR was pointing to Drive.
CORRECTED_IMG_DIR = "/content/coco_dataset/train2014"

def load_image(path):
    # Use CORRECTED_IMG_DIR for the path where images are actually located
    full_path = tf.strings.join([CORRECTED_IMG_DIR, path], separator='/')
    img = tf.io.read_file(full_path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, (224, 224))
    img = tf.keras.applications.resnet50.preprocess_input(img)
    return img

train_dataset = tf.data.Dataset.from_tensor_slices(
    (img_paths_flat, input_seqs, target_seqs)
)

train_dataset = train_dataset.shuffle(1000)

train_dataset = train_dataset.map(
    lambda p, inp, tar: (load_image(p), inp, tar),
    num_parallel_calls=tf.data.AUTOTUNE
)

train_dataset = train_dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

# Sanity check
for imgs, inp, tar in train_dataset.take(1):
    print("Images:", imgs.shape)
    print("Input captions:", inp.shape)
    print("Target captions:", tar.shape)

Images: (16, 224, 224, 3)
Input captions: (16, 19)
Target captions: (16, 19)


In [ ]:
import os
import json
import pickle
from tensorflow.keras.preprocessing.text import Tokenizer

# Paths
TOKENIZER_PATH = "/content/drive/MyDrive/tokenizer.pkl"
ANN_DIR = "/content/drive/MyDrive/coco_dataset/annotations"

# 1️⃣ Load COCO captions
with open(os.path.join(ANN_DIR, "captions_train2014.json"), "r") as f:
    coco_train = json.load(f)

captions_to_fit = []
for ann in coco_train["annotations"]:
    captions_to_fit.append(f"<start> {ann['caption']} <end>")

# 2️⃣ Create tokenizer (include all words)
tokenizer = Tokenizer(
    oov_token="<unk>",  # unknown words mapped to <unk>
    filters=''          # do not remove < or >
)
tokenizer.fit_on_texts(captions_to_fit)

# 3️⃣ Ensure <pad> token exists at index 0
if '<pad>' not in tokenizer.word_index:
    word_index_shifted = {word: idx + 1 for word, idx in tokenizer.word_index.items()}
    word_index_shifted['<pad>'] = 0
    tokenizer.word_index = word_index_shifted
    tokenizer.index_word = {idx: word for word, idx in tokenizer.word_index.items()}

# 4️⃣ Final VOCAB_SIZE = embedding size expected by decoder
VOCAB_SIZE = len(tokenizer.word_index) + 1  # +1 if you want a reserved zero

# 5️⃣ Save tokenizer
with open(TOKENIZER_PATH, "wb") as f:
    pickle.dump(tokenizer, f)

MAX_LEN = 20
print(f"Tokenizer created/loaded. VOCAB_SIZE: {VOCAB_SIZE}")
print(f"MAX_LEN: {MAX_LEN}")


Tokenizer created/loaded. VOCAB_SIZE: 37739
MAX_LEN: 20


In [ ]:
base_model = tf.keras.applications.ResNet50(
    include_top=False, weights="imagenet"
)
base_model.trainable = False

class Encoder(tf.keras.Model):
    def __init__(self):
        super().__init__()
        self.cnn = base_model
        self.fc = tf.keras.layers.Dense(256)

    def call(self, x):
        x = self.cnn(x)
        x = tf.reshape(x, (x.shape[0], -1, x.shape[-1]))
        x = self.fc(x)
        return x


In [ ]:
train_dataset = train_dataset.shuffle(256)


In [ ]:
base_model.trainable = False


In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, Model

class BahdanauAttention(tf.keras.layers.Layer):
    def __init__(self, units):
        super().__init__()
        self.W1 = tf.keras.layers.Dense(units)
        self.W2 = tf.keras.layers.Dense(units)
        self.V  = tf.keras.layers.Dense(1)

    def call(self, features, hidden):
        # features: (batch, 49, embed_dim)
        # hidden:   (batch, units)
        hidden_with_time = tf.expand_dims(hidden, 1)   # (batch,1,units)
        score = self.V(tf.nn.tanh(self.W1(features) + self.W2(hidden_with_time)))
        weights = tf.nn.softmax(score, axis=1)
        context = tf.reduce_sum(weights * features, axis=1)  # (batch, embed_dim)
        return context

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, Model

class CNN_Encoder(Model):
    def __init__(self, embed_dim):
        super().__init__()
        base = tf.keras.applications.ResNet50(
            include_top=False,
            weights='imagenet'
        )
        base.trainable = False
        self.cnn = Model(base.input, base.output)
        self.fc  = layers.Dense(embed_dim)

    def call(self, images):
        x = self.cnn(images) # Output (batch, H, W, Channels) e.g., (16, 7, 7, 2048)

        # Get dynamic shapes
        batch_size = tf.shape(x)[0]
        height = tf.shape(x)[1]
        width = tf.shape(x)[2]
        channels = tf.shape(x)[3] # This is 2048

        # Reshape to 2D for the Dense layer: (batch_size * num_patches, channels)
        # num_patches = height * width
        x_reshaped_for_dense = tf.reshape(x, (batch_size * height * width, channels))

        # Apply Dense layer
        x_dense_output = self.fc(x_reshaped_for_dense)

        # Reshape back to 3D: (batch_size, num_patches, embed_dim)
        x_final = tf.reshape(x_dense_output, (batch_size, height * width, self.fc.units))
        return x_final

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, Model

class BahdanauAttention(tf.keras.layers.Layer):
    def __init__(self, units):
        super().__init__()
        self.W1 = tf.keras.layers.Dense(units)
        self.W2 = tf.keras.layers.Dense(units)
        self.V  = tf.keras.layers.Dense(1)

    def call(self, features, hidden):
        # features: (batch, 49, embed_dim)
        # hidden:   (batch, units)
        hidden_with_time = tf.expand_dims(hidden, 1)   # (batch,1,units)
        score = self.V(tf.nn.tanh(self.W1(features) + self.W2(hidden_with_time)))
        weights = tf.nn.softmax(score, axis=1)
        context = tf.reduce_sum(weights * features, axis=1)  # (batch, embed_dim)
        return context

class RNN_Decoder(tf.keras.Model):
    def __init__(self, vocab_size, embed_dim, units):
        super().__init__()
        self.units = units
        self.embedding = tf.keras.layers.Embedding(vocab_size, embed_dim)
        self.attention = BahdanauAttention(units)
        self.lstm = tf.keras.layers.LSTM(
            units,
            return_sequences=True,
            return_state=True
        )
        self.fc = tf.keras.layers.Dense(vocab_size)

    def call(self, x, features, hidden, cell=None, training=True):
        if cell is None:
            cell = tf.zeros_like(hidden)

        x = self.embedding(x)
        if training:
            context = self.attention(features, hidden)
            context = tf.expand_dims(context, 1)
            context = tf.tile(context, [1, x.shape[1], 1])
        else:
            context = self.attention(features, hidden)
            context = tf.expand_dims(context, 1)

        x = tf.concat([context, x], axis=-1)
        output, h, c = self.lstm(x, initial_state=[hidden, cell])
        output = tf.reshape(output, (-1, output.shape[2]))
        x = self.fc(output)
        return x, h, c

    def reset_state(self, batch_size):
        return tf.zeros((batch_size, self.units))

EMBED_DIM = 256
UNITS     = 512
# VOCAB_SIZE is now correctly defined globally in Cozu5Pb2OR9H as 37738
# Do NOT re-calculate it here using len(tokenizer.word_index) + 1

encoder = CNN_Encoder(EMBED_DIM)
decoder = RNN_Decoder(VOCAB_SIZE, EMBED_DIM, UNITS)

loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True, reduction='none')
optimizer = tf.keras.optimizers.Adam(learning_rate=1e-4)

def loss_function(real, pred):
    mask = tf.math.not_equal(real, 0)
    loss = loss_fn(real, pred)
    mask = tf.cast(mask, loss.dtype)
    loss *= mask
    return tf.reduce_mean(loss)

In [ ]:
BATCH_SIZE = 16
EMBED_DIM = 256
UNITS = 512
IMG_SIZE = 224


In [ ]:
EMBED_DIM = 256
UNITS = 512

encoder = CNN_Encoder(EMBED_DIM)
decoder = RNN_Decoder(vocab_size=VOCAB_SIZE, embed_dim=EMBED_DIM, units=UNITS)


In [ ]:
decoder = RNN_Decoder(VOCAB_SIZE, EMBED_DIM, UNITS)


In [ ]:
import tensorflow as tf#yo

@tf.function
def train_step(images, input_captions, target_captions):
    loss = 0.0
    batch_size = tf.shape(images)[0]
    hidden = decoder.reset_state(batch_size)
    cell = tf.zeros((batch_size, UNITS))

    with tf.GradientTape() as tape:
        features = encoder(images)  # (batch,49,embed_dim)
        for t in range(input_captions.shape[1]): # Iterate over length of input captions
            dec_input = tf.expand_dims(input_captions[:, t], 1)  # (batch,1)
            preds, hidden, cell = decoder(dec_input, features, hidden, cell)
            loss += loss_function(target_captions[:, t], preds) # Use target_captions for loss

    trainable_vars = encoder.trainable_variables + decoder.trainable_variables
    grads = tape.gradient(loss, trainable_vars)

    # Filter out None gradients
    grads_and_vars = []
    for grad, var in zip(grads, trainable_vars):
        if grad is not None:
            grads_and_vars.append((grad, var))

    optimizer.apply_gradients(grads_and_vars)

    return loss / tf.cast(input_captions.shape[1], tf.float32)

In [ ]:
import os
import tensorflow as tf

# Directory where checkpoints will be saved
checkpoint_dir = "/content/drive/MyDrive/coco_checkpoints"
os.makedirs(checkpoint_dir, exist_ok=True)

checkpoint_prefix = os.path.join(checkpoint_dir, "ckpt")

# Create checkpoint object
ckpt = tf.train.Checkpoint(
    encoder=encoder,
    decoder=decoder,
    optimizer=optimizer
)

# Checkpoint manager (keeps last N checkpoints)
ckpt_manager = tf.train.CheckpointManager(
    ckpt,
    checkpoint_dir,
    max_to_keep=5
)


In [ ]:
checkpoint_dir = '/content/drive/MyDrive/coco_checkpoints'


In [ ]:
# Create checkpoint object (must match training)
ckpt = tf.train.Checkpoint(
    encoder=encoder,
    decoder=decoder,
    optimizer=optimizer  # optional but recommended
)

# Checkpoint manager
ckpt_manager = tf.train.CheckpointManager(
    ckpt,
    checkpoint_dir,
    max_to_keep=5
)

# Inspect checkpoints
print("Valid checkpoints found:")
for ck in ckpt_manager.checkpoints:
    print(ck)

print("Latest checkpoint:", ckpt_manager.latest_checkpoint)

# Restore
if ckpt_manager.latest_checkpoint:
    ckpt.restore(ckpt_manager.latest_checkpoint).expect_partial()
    print("Restored from", ckpt_manager.latest_checkpoint)
else:
    print("No checkpoint found, training from scratch")


Valid checkpoints found:
/content/drive/MyDrive/coco_checkpoints/ckpt-97
/content/drive/MyDrive/coco_checkpoints/ckpt-98
/content/drive/MyDrive/coco_checkpoints/ckpt-99
/content/drive/MyDrive/coco_checkpoints/ckpt-100
/content/drive/MyDrive/coco_checkpoints/ckpt-101
Latest checkpoint: /content/drive/MyDrive/coco_checkpoints/ckpt-101
Restored from /content/drive/MyDrive/coco_checkpoints/ckpt-101


In [ ]:
print("Encoder trainable vars:", len(encoder.trainable_variables))
print("Decoder trainable vars:", len(decoder.trainable_variables))


Encoder trainable vars: 0
Decoder trainable vars: 0


In [ ]:
# Ensure trainable is correct
encoder.trainable = False
decoder.trainable = True

# Make a dummy forward pass to build decoder variables
dummy_img = tf.zeros((1, 224, 224, 3))
features = encoder(dummy_img)

start_token = tokenizer.word_index["<start>"]
dummy_token = tf.constant([start_token])
x = decoder.embedding(dummy_token)

h = decoder.reset_state(1)
c = tf.zeros((1, UNITS))

_ = decoder(x, features, h, c)  # this builds the variables

# Verify
print("Encoder vars:", len(encoder.trainable_variables))  # should be 0
print("Decoder vars:", len(decoder.trainable_variables))  # should now be >0


Encoder vars: 0
Decoder vars: 16


In [ ]:
import os #NEWerror
import tensorflow as tf

# -------------------------------
# PARAMETERS
# -------------------------------
EPOCHS = 22                 # total epochs
BATCH_SAVE = 500            # save checkpoint every N batches
CHECKPOINT_DIR = "/content/drive/MyDrive/coco_checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# -------------------------------
# ENSURE TRAINABLE VARS
# -------------------------------
encoder.trainable = False
decoder.trainable = True  # MUST be set BEFORE optimizer

# -------------------------------
# REBUILD OPTIMIZER
# -------------------------------
optimizer = tf.keras.optimizers.Adam()

# -------------------------------
# BUILD CHECKPOINT
# -------------------------------
ckpt = tf.train.Checkpoint(encoder=encoder, decoder=decoder, optimizer=optimizer)
ckpt_manager = tf.train.CheckpointManager(ckpt, CHECKPOINT_DIR, max_to_keep=5)

# -------------------------------
# DUMMY FORWARD PASS TO BUILD DECODER VARIABLES
# -------------------------------
BATCH_SIZE = 16
IMG_SIZE = 224
EMBED_DIM = 256
UNITS = 512

dummy_images = tf.random.uniform([BATCH_SIZE, IMG_SIZE, IMG_SIZE, 3])
_ = encoder(dummy_images)

dummy_input_token = tf.zeros([BATCH_SIZE, 1], dtype=tf.int32)
dummy_features = tf.random.uniform([BATCH_SIZE, 49, EMBED_DIM])
dummy_hidden = tf.zeros([BATCH_SIZE, UNITS])
dummy_cell = tf.zeros([BATCH_SIZE, UNITS])
_ = decoder(dummy_input_token, dummy_features, dummy_hidden, dummy_cell)

print(f"Encoder vars: {len(encoder.trainable_variables)}")  # should be 0
print(f"Decoder vars: {len(decoder.trainable_variables)}")  # should now be >0

# -------------------------------
# RESTORE CHECKPOINT (weights only)
# -------------------------------
if ckpt_manager.latest_checkpoint:
    status = ckpt.restore(ckpt_manager.latest_checkpoint)
    status.expect_partial()  # ignore unmatched vars warning
    print(f"Restored from checkpoint: {ckpt_manager.latest_checkpoint}")
else:
    print("No checkpoint found. Training from scratch.")

# FORCE optimizer variable creation ONCE
optimizer.build(decoder.trainable_variables)


# -------------------------------
# TRAINING LOOP
# -------------------------------

last_epoch_file = os.path.join(CHECKPOINT_DIR, "last_epoch.txt")

start_epoch = 0
if os.path.exists(last_epoch_file):
    start_epoch = int(open(last_epoch_file).read()) + 1

print(f"Starting from epoch {start_epoch + 1}")

for epoch in range(start_epoch, EPOCHS):
    total_loss = 0.0

    for batch, (imgs, inp, tar) in enumerate(train_dataset):
        batch_loss = train_step(imgs, inp, tar)
        total_loss += batch_loss

        if (batch + 1) % 50 == 0:
            print(f"Epoch {epoch+1} Batch {batch+1} Loss {float(batch_loss):.4f}")

    avg_loss = total_loss / (batch + 1)
    print(f"Epoch {epoch+1} completed. Average Loss: {avg_loss:.4f}")

    ckpt_manager.save()
    with open(last_epoch_file, "w") as f:
        f.write(str(epoch))

last_epoch_file = os.path.join(CHECKPOINT_DIR, "last_epoch.txt")
last_batch_file = os.path.join(CHECKPOINT_DIR, "last_batch.txt")

start_epoch = int(open(last_epoch_file).read()) if os.path.exists(last_epoch_file) else 0

num_batches = tf.data.experimental.cardinality(train_dataset).numpy()
print(f"Total batches in dataset: {num_batches}")

for epoch in range(start_epoch, EPOCHS):
    total_loss = 0.0

    # Reset batch numbering for logging
    for batch, (imgs, inp, tar) in enumerate(train_dataset):
        batch_loss = train_step(imgs, inp, tar)
        total_loss += batch_loss

        # Print every 50 batches
        if (batch + 1) % 50 == 0:
            print(f"Epoch {epoch+1} Batch {batch+1} Loss {float(batch_loss):.4f}")

        # Save checkpoint every BATCH_SAVE batches
        if (batch + 1) % BATCH_SAVE == 0:
            ckpt_manager.save()
            print(f"Checkpoint saved at batch {batch+1}")
            with open(last_epoch_file, "w") as f:
                f.write(str(epoch))
            with open(last_batch_file, "w") as f:
                f.write(str(batch+1))

    # End of epoch
    avg_loss = float(total_loss / (batch + 1))
    print(f"Epoch {epoch+1} completed. Average Loss: {avg_loss:.4f}")

    # Save checkpoint at end of epoch
    ckpt_manager.save()
    print(f"Checkpoint saved at end of epoch {epoch+1}")

    # Reset last batch for next epoch
    with open(last_epoch_file, "w") as f:
        f.write(str(epoch))
    with open(last_batch_file, "w") as f:
        f.write("0")  # batch numbering resets

print("Training complete.")


OUTPUT GENERATION

In [ ]:
# ----------------------------
# IMAGE CAPTIONING (GREEDY SEARCH)
# ----------------------------
import pickle
import tensorflow as tf
from tensorflow.keras.preprocessing import image as keras_image
from google.colab import files
import numpy as np

# 1️⃣ Upload image
uploaded = files.upload()
img_path = next(iter(uploaded.keys()))
print("Uploaded image:", img_path)

# 2️⃣ Preprocess image
def load_image(img_path):
    img = keras_image.load_img(img_path, target_size=(224, 224))
    img = keras_image.img_to_array(img)
    img = tf.keras.applications.resnet50.preprocess_input(img)
    return tf.convert_to_tensor(img, dtype=tf.float32)

img_tensor = load_image(img_path)
print("Image preprocessed:", img_tensor.shape)
def greedy_caption(encoder, decoder, image, tokenizer, max_length=20):
    image = tf.expand_dims(image, 0)
    features = encoder(image)

    start_token = tokenizer.word_index["<start>"]
    end_token = tokenizer.word_index["<end>"]

    x = tf.constant([[start_token]], dtype=tf.int32)
    h = decoder.reset_state(batch_size=1)

    result = []

    for step in range(max_length):
        preds, h, _ = decoder(x, features, h)  # IGNORE attention
        predicted_id = int(tf.argmax(preds[0]).numpy())
        word = tokenizer.index_word.get(predicted_id, "")

        print(f"step {step} | id {predicted_id} | word '{word}'")

        if predicted_id == end_token:
            break

        result.append(word)
        x = tf.constant([[predicted_id]], dtype=tf.int32)

    return " ".join(result)


# 4️⃣ Generate caption using greedy decoding
caption = greedy_caption(
    encoder=encoder,
    decoder=decoder,
    image=img_tensor,    # your preprocessed image tensor (224x224x3)
    tokenizer=tokenizer,
    max_length=20        # maximum words in the caption
)

print("Generated caption:", caption)


In [ ]:
# -----------------------------
# IMAGE CAPTIONING (BEAM SEARCH)
# -----------------------------
import tensorflow as tf
from tensorflow.keras.preprocessing import image as keras_image
from google.colab import files
import numpy as np

# -----------------------------
# 1️⃣ Upload image
# -----------------------------
uploaded = files.upload()
img_path = next(iter(uploaded.keys()))
print("Uploaded image:", img_path)

# -----------------------------
# 2️⃣ Preprocess image
# -----------------------------
def load_image(img_path):
    img = keras_image.load_img(img_path, target_size=(224, 224))
    img = keras_image.img_to_array(img)
    img = tf.keras.applications.resnet50.preprocess_input(img)
    return tf.convert_to_tensor(img, dtype=tf.float32)

img_tensor = load_image(img_path)
img_tensor = tf.expand_dims(img_tensor, 0)  # shape (1, 224, 224, 3)
print("Image preprocessed:", img_tensor.shape)

# -----------------------------
# 3️⃣ Beam Search Caption
# -----------------------------
def beam_search_caption(encoder, decoder, image, tokenizer, max_length=20, beam_width=3, repetition_penalty=1.2):
    # 1️⃣ Encode image
    features = encoder(image)  # (1, 49, 256)

    # 2️⃣ Beam search setup
    start_token = tokenizer.word_index["<start>"]
    end_token = tokenizer.word_index["<end>"]

    sequences = [[ [start_token], 0.0, tf.zeros((1, decoder.units)), tf.zeros((1, decoder.units)) ]]  # [sequence, score, hidden, cell]

    for _ in range(max_length):
        all_candidates = []

        for seq, score, hidden, cell in sequences:
            if seq[-1] == end_token:
                all_candidates.append([seq, score, hidden, cell])
                continue

            x = tf.constant([[seq[-1]]], dtype=tf.int32)
            x = decoder.embedding(x)  # (1, 1, embed_dim)

            # ✅ compute context correctly
            context = decoder.attention(features, hidden)  # (1, units)
            if len(context.shape) == 2:
                context = tf.expand_dims(context, 1)  # (1, 1, units)

            x_input = tf.concat([context, x], axis=-1)  # (1, 1, units+embed_dim)
            output, h, c = decoder.lstm(x_input, initial_state=[hidden, cell])
            output = tf.reshape(output, (-1, output.shape[2]))  # (1, lstm_units)
            preds = decoder.fc(output)  # (1, vocab_size)
            preds = tf.nn.softmax(preds, axis=-1)

            # ✅ repetition penalty
            preds = tf.squeeze(preds, axis=0).numpy()  # (vocab_size,)
            for word_id in set(seq):
                preds[word_id] /= repetition_penalty

            top_ids = preds.argsort()[-beam_width:][::-1]

            for word_id in top_ids:
                candidate = [seq + [word_id], score - np.log(preds[word_id]+1e-9), h, c]
                all_candidates.append(candidate)

        # select best beam_width sequences
        sequences = sorted(all_candidates, key=lambda x: x[1])[:beam_width]

    # return the best sequence
    best_seq = sequences[0][0]
    words = [tokenizer.index_word.get(id, "") for id in best_seq if id not in [start_token, end_token]]
    return " ".join(words)

# -----------------------------
# 4️⃣ Generate caption
# -----------------------------
caption = beam_search_caption(
    encoder=encoder,
    decoder=decoder,
    image=img_tensor,
    tokenizer=tokenizer,
    max_length=20,
    beam_width=3,
    repetition_penalty=1.2
)

print("Beam search caption:", caption)


In [ ]:
print(len(tokenizer.word_index))
print(tokenizer.word_index.get("<start>"))

37738
4


In [ ]:
decoder.embedding.weights[0].shape

TensorShape([37739, 256])

In [ ]:
print(tf.reduce_sum(decoder.trainable_variables[0]))

tf.Tensor(8401.75, shape=(), dtype=float32)


In [ ]:
print(tf.train.latest_checkpoint(checkpoint_dir))

/content/drive/MyDrive/coco_checkpoints/ckpt-101


BLEU SCORE EVALUATION

In [ ]:
!ls /content


coco_dataset  COCO_train2014_000000291366.jpg  drive  sample_data


In [ ]:
annotation_path = "/content/coco_dataset/annotations/captions_train2014.json"


In [ ]:
import os
print(os.path.exists(annotation_path))


True


In [ ]:
!pip install nltk
import nltk
nltk.download('punkt')


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [ ]:
import json
import os
import numpy as np
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction


In [ ]:
VAL_IMAGE_DIR = "/content/coco_dataset/val2014"
VAL_ANN_PATH = "/content/coco_dataset/annotations/captions_val2014.json"


In [ ]:
with open(VAL_ANN_PATH, "r") as f:
    coco_data = json.load(f)

# image_id → list of captions
references = {}
for ann in coco_data["annotations"]:
    img_id = ann["image_id"]
    caption = ann["caption"].lower().strip()
    references.setdefault(img_id, []).append(caption)


In [ ]:
id_to_filename = {
    img["id"]: img["file_name"]
    for img in coco_data["images"]
}


In [ ]:
def evaluate_bleu(
    encoder,
    decoder,
    tokenizer,
    image_dir,
    references,
    id_to_filename,
    max_samples=100
):
    smoothie = SmoothingFunction().method4
    all_references = []
    all_hypotheses = []

    image_ids = list(references.keys())[:max_samples]

    for img_id in image_ids:
        img_path = os.path.join(image_dir, id_to_filename[img_id])

        if not os.path.exists(img_path):
            continue

        # generate caption
        caption = beam_search_caption(
            encoder=encoder,
            decoder=decoder,
            image=img_tensor,
            tokenizer=tokenizer,
            max_length=20,
            beam_width=3
        )

        # tokenize
        hypothesis = caption.lower().split()
        ref_tokens = [ref.split() for ref in references[img_id]]

        all_hypotheses.append(hypothesis)
        all_references.append(ref_tokens)

    bleu1 = corpus_bleu(all_references, all_hypotheses,
                        weights=(1, 0, 0, 0), smoothing_function=smoothie)
    bleu2 = corpus_bleu(all_references, all_hypotheses,
                        weights=(0.5, 0.5, 0, 0), smoothing_function=smoothie)
    bleu3 = corpus_bleu(all_references, all_hypotheses,
                        weights=(0.33, 0.33, 0.33, 0), smoothing_function=smoothie)
    bleu4 = corpus_bleu(all_references, all_hypotheses,
                        weights=(0.25, 0.25, 0.25, 0.25), smoothing_function=smoothie)

    return bleu1, bleu2, bleu3, bleu4


In [ ]:
bleu1, bleu2, bleu3, bleu4 = evaluate_bleu(
    encoder,
    decoder,
    tokenizer,
    VAL_IMAGE_DIR,
    references,
    id_to_filename,
    max_samples=100   # start small
)

print(f"BLEU-1: {bleu1:.4f}")
print(f"BLEU-2: {bleu2:.4f}")
print(f"BLEU-3: {bleu3:.4f}")
print(f"BLEU-4: {bleu4:.4f}")


BLEU-1: 0.4411
BLEU-2: 0.1644
BLEU-3: 0.0552
BLEU-4: 0.0268


In [ ]:
# ----------------------------
# IMAGE CAPTIONING (BEAM SEARCH)
# ----------------------------
import pickle
import tensorflow as tf
from tensorflow.keras.preprocessing import image as keras_image
from google.colab import files
import numpy as np

# 1️⃣ Upload image
uploaded = files.upload()
img_path = next(iter(uploaded.keys()))
print("Uploaded image:", img_path)

# 2️⃣ Preprocess image
def load_image(img_path):
    img = keras_image.load_img(img_path, target_size=(224, 224))
    img = keras_image.img_to_array(img)
    img = tf.keras.applications.resnet50.preprocess_input(img)
    return tf.convert_to_tensor(img, dtype=tf.float32)

img_tensor = load_image(img_path)
print("Image preprocessed:", img_tensor.shape)


# 3️⃣ Beam Search Function (FIXED)
def generate_caption_beam_search(image, encoder, decoder, tokenizer, max_len=20, beam_width=3):
    image = tf.expand_dims(image, 0)
    features = encoder(image)  # (1, num_patches, embed_dim)

    start_token = tokenizer.word_index.get("<start>") or tokenizer.word_index.get("startseq")
    end_token = tokenizer.word_index.get("<end>") or tokenizer.word_index.get("endseq")
    if start_token is None:
        raise ValueError("Start token not found in tokenizer")

    hidden = tf.zeros((1, decoder.units))
    cell = tf.zeros((1, decoder.units))

    # Beam: (sequence, score, hidden, cell)
    beams = [([start_token], 0.0, hidden, cell)]
    completed = []

    for _ in range(max_len):
        new_beams = []
        for seq, score, hidden, cell in beams:
            last_token = seq[-1]
            if last_token == end_token:
                completed.append((seq, score))
                continue

            # ✅ Pass token IDs directly, do NOT embed manually
            dec_input = tf.expand_dims([last_token], 1)  # (1,1)
            preds, new_hidden, new_cell = decoder(dec_input, features, hidden, cell, training=False)

            # Use log probabilities
            probs = tf.nn.log_softmax(preds[0]).numpy()  # shape=(vocab_size,)
            top_k = np.argsort(probs)[-beam_width:]

            for idx in top_k:
                if len(seq) > 1 and idx == seq[-1]:  # prevent immediate repetition
                    continue
                new_seq = seq + [idx]
                new_score = score + probs[idx]
                new_beams.append((new_seq, new_score, new_hidden, new_cell))

        beams = sorted(new_beams, key=lambda x: x[1], reverse=True)[:beam_width]
        if not beams:
            break

    completed += [(seq, score) for seq, score, _, _ in beams]
    best_seq = max(completed, key=lambda x: x[1])[0]

    words = [tokenizer.index_word.get(idx, "<unk>") for idx in best_seq if idx not in (start_token, end_token)]
    return " ".join(words)


# 4️⃣ Generate caption
caption = generate_caption_beam_search(
    image=img_tensor,
    encoder=encoder,
    decoder=decoder,
    tokenizer=tokenizer,
    max_len=20,
    beam_width=3
)

print("Generated Caption:", caption)


Saving COCO_train2014_000000000081.jpg to COCO_train2014_000000000081.jpg
Uploaded image: COCO_train2014_000000000081.jpg
Image preprocessed: (224, 224, 3)
Generated Caption: a person laying on the side of a car.


In [ ]:
import os
import tensorflow as tf # Added import for tf

# Training parameters
EPOCHS = 10
BATCH_SAVE = 500
CHECKPOINT_DIR = "/content/drive/MyDrive/coco_checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# Checkpoint manager
ckpt = tf.train.Checkpoint(encoder=encoder, decoder=decoder, optimizer=optimizer)
ckpt_manager = tf.train.CheckpointManager(ckpt, CHECKPOINT_DIR, max_to_keep=3)

# Force build models and create trainable variables
# Encoder dummy call
dummy_images = tf.random.uniform([BATCH_SIZE, IMG_SIZE, IMG_SIZE, 3])
_ = encoder(dummy_images)

# Decoder dummy call
# Needs one input token, features, hidden, and cell state
dummy_input_token = tf.zeros([BATCH_SIZE, 1], dtype=tf.int32)
dummy_features = tf.random.uniform([BATCH_SIZE, 49, EMBED_DIM]) # 49 patches from ResNet50 output
dummy_hidden = tf.zeros([BATCH_SIZE, UNITS])
dummy_cell = tf.zeros([BATCH_SIZE, UNITS])
_ = decoder(dummy_input_token, dummy_features, dummy_hidden, dummy_cell)

# Now, re-check trainable variables (optional, but good for verification)
print("Encoder trainable variables after dummy call:", len(encoder.trainable_variables))
print("Decoder trainable variables after dummy call:", len(decoder.trainable_variables))

for epoch in range(EPOCHS):
    total_loss = 0.0 # Initialize as float
    for batch, (imgs, inp, tar) in enumerate(train_dataset):
        batch_loss = train_step(imgs, inp, tar)
        total_loss += batch_loss

        if batch % 100 == 0:
            print(f"Epoch {epoch+1} Batch {batch} Loss {float(batch_loss):.4f}")

        if (batch + 1) % BATCH_SAVE == 0:
            ckpt_manager.save()
            print(f"Checkpoint saved at batch {batch+1}")

        # Optional: shorten epoch for testing
        if batch + 1 >= 1000:
            print(f"Ending epoch {epoch+1} early at batch {batch+1}")
            break

    print(f"Epoch {epoch+1} Loss {float(total_loss / (batch+1)):.4f}")
    ckpt_manager.save()
    print(f"Checkpoint saved at end of epoch {epoch+1}")

Encoder trainable variables after dummy call: 2
Decoder trainable variables after dummy call: 12
Epoch 1 Batch 0 Loss 0.9832
Epoch 1 Batch 100 Loss 1.4135
Epoch 1 Batch 200 Loss 0.9866
Epoch 1 Batch 300 Loss 1.0340
Epoch 1 Batch 400 Loss 1.0793
Checkpoint saved at batch 500
Epoch 1 Batch 500 Loss 1.1224
Epoch 1 Batch 600 Loss 1.2776
Epoch 1 Batch 700 Loss 1.2522
Epoch 1 Batch 800 Loss 0.9937
Epoch 1 Batch 900 Loss 1.2302
Checkpoint saved at batch 1000
Ending epoch 1 early at batch 1000
Epoch 1 Loss 1.1523
Checkpoint saved at end of epoch 1
Epoch 2 Batch 0 Loss 0.9872
Epoch 2 Batch 100 Loss 0.8981
Epoch 2 Batch 200 Loss 0.7763
Epoch 2 Batch 300 Loss 0.8833
Epoch 2 Batch 400 Loss 1.0978
Checkpoint saved at batch 500
Epoch 2 Batch 500 Loss 0.8103
Epoch 2 Batch 600 Loss 0.8003
Epoch 2 Batch 700 Loss 1.1347
Epoch 2 Batch 800 Loss 0.9779
Epoch 2 Batch 900 Loss 1.0888
Checkpoint saved at batch 1000
Ending epoch 2 early at batch 1000
Epoch 2 Loss 1.0330
Checkpoint saved at end of epoch 2
Epoch